### What i want to discover first:
- Total cost of importing from each country
- Total cost of importing from each supplier (if possible)

Starting from the point that the imported quantity or the Dollar/Kilo price of a product does not affect any data about another product, let us first discover the price of one commodity and then of the other.

#### Sadly i found 3 free suppliers (DataWeb, ImportYeti and Volza), i need to merge them.
I have to take to account that both ImportYeti and Volza take data from the same source

## 1. Setup
Three cells: every import in the notebook, the package upgrade, then Chrome discovery plus the Python 3.14 `distutils` shim that `undetected_chromedriver` still expects. `version_main` is pinned to the installed Chrome major, otherwise the driver download guesses wrong.

`undetected_chromedriver` is the one import left out of the import cell — it runs `from distutils.version import LooseVersion` at import time, so it can only load after the shim, and stays inside `create_stealth_driver()`.


In [37]:
# Todos os imports do caderno ficam nesta celula. Ela roda antes do pip da
# celula seguinte: um upgrade de selenium so vale depois de reiniciar o kernel,
# porque reinstalar nao recarrega um modulo ja importado.
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
import types
import urllib.request
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from selenium.common.exceptions import (
    NoSuchElementException, StaleElementReferenceException, TimeoutException,
)
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# undetected_chromedriver esta fora desta celula de proposito: ele faz
# `from distutils.version import LooseVersion` no proprio import, entao so pode
# ser carregado depois do shim da celula seguinte. Fica dentro de
# create_stealth_driver().


In [38]:
print("Verifying/upgrading packages for Python 3.14 compatibility...")

packages_to_upgrade = [
    'undetected-chromedriver',
    'selenium',
]

for package in packages_to_upgrade:
    try:
        # Correct pip syntax: --upgrade is a separate argument
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            "-q", "--upgrade", package
        ])
        print(f"✓ {package} is up to date")
    except Exception as e:
        print(f"⚠ {package}: {str(e)[:80]}")

print("\n✓ Setup complete! Ready for web scraping")
print("  Se algum pacote foi de fato atualizado agora, reinicie o kernel:\n   selenium ja esta carregado e continuaria na versao antiga.")


Verifying/upgrading packages for Python 3.14 compatibility...
✓ undetected-chromedriver is up to date
✓ selenium is up to date

✓ Setup complete! Ready for web scraping
  Se algum pacote foi de fato atualizado agora, reinicie o kernel:
   selenium ja esta carregado e continuaria na versao antiga.


In [ ]:
warnings.filterwarnings('ignore')


class LooseVersion: #configurar o que falta no distutils, que foi removido pelo python 

    def __init__(self, vstring="0"):
        self.vstring = vstring
        self.version = self._parse(vstring)

    def _parse(self, vstring):
        try:
            return [int(part) for part in vstring.split('.')]
        except ValueError:
            return [0]

    def __lt__(self, other): return self.version < other.version
    def __le__(self, other): return self.version <= other.version
    def __gt__(self, other): return self.version > other.version
    def __ge__(self, other): return self.version >= other.version
    def __eq__(self, other): return self.version == other.version
    def __repr__(self): return f"LooseVersion('{self.vstring}')"

if sys.version_info >= (3, 12) and 'distutils.version' not in sys.modules:
    _distutils_version = types.ModuleType('distutils.version')
    _distutils_version.LooseVersion = LooseVersion
    _distutils = types.ModuleType('distutils')
    _distutils.version = _distutils_version
    sys.modules['distutils.version'] = _distutils_version
    sys.modules['distutils'] = _distutils

def human_time_sleep(seconds):
    time.sleep(seconds * random.uniform(0.9, 1.1))

def find_chrome_binary():
    for name in ('google-chrome', 'google-chrome-stable', 'chromium', 'chromium-browser', 'chrome'):
        path = shutil.which(name)
        if path:
            return path
    for path in ('/usr/bin/google-chrome', '/usr/bin/chromium', '/snap/bin/chromium'):
        if os.path.exists(path):
            return path
    return None


def preprocess_version(chrome_binary):
    try:
        out = subprocess.check_output([chrome_binary, '--version'], text=True)
        match = re.search(r'(\d+)\.', out) # se a versão for 141.0.7390.54 vai para 141
        return int(match.group(1)) if match else None
    except Exception:
        return None


CHROME_BINARY = find_chrome_binary()
PROCESSED_VERSION = preprocess_version(CHROME_BINARY) if CHROME_BINARY else None
print(f"Chrome binary: {CHROME_BINARY}")
print(f"Chrome version: {PROCESSED_VERSION}")


Chrome binary: /usr/bin/google-chrome
Chrome major version: 151


## 2. Browser session
The profile is cloned so a second Chrome inherits the logged-in session; locks and caches are skipped because Chrome refuses to start on a user-data-dir holding another instance's `SingletonLock`.

In [ ]:
CHROME_BASE_DIR = os.path.expanduser("~/.config/google-chrome")
CHROME_PROFILE_CLONE_DIR = os.path.join(os.getcwd(), ".chrome_profile_clone")

DONT_COPY_THIS_WHEN_CLONNING_GOOGLE = shutil.ignore_patterns(
    "SingletonLock", "SingletonCookie", "SingletonSocket", "lockfile",
    "Cache", "Code Cache", "GPUCache", "ShaderCache", "GrShaderCache",
    "Crash Reports", "Crashpad",
)

STEALTH_JS = """
delete Object.getPrototypeOf(navigator).webdriver;
window.chrome = window.chrome || { runtime: {} };
Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]});
Object.defineProperty(navigator, 'languages', {get: () => ['en-US', 'en']});
const _getParameter = WebGLRenderingContext.prototype.getParameter;
WebGLRenderingContext.prototype.getParameter = function(parameter) {
    if (parameter === 37445) return 'Intel Inc.';
    if (parameter === 37446) return 'Intel Iris OpenGL Engine';
    return _getParameter.apply(this, arguments);
};
const _permissionsQuery = navigator.permissions.query;
navigator.permissions.query = (parameters) => (
    parameters.name === 'notifications'
        ? Promise.resolve({state: Notification.permission})
        : _permissionsQuery(parameters)
);
"""

CHROME_PROFILE = "Default"


In [ ]:

def clone_chrome_profile():
    if not os.path.isdir(CHROME_BASE_DIR):
        raise FileNotFoundError(f"{CHROME_BASE_DIR} not found. Open Chrome, log in, then close it fully.")
    if not os.path.isdir(os.path.join(CHROME_BASE_DIR, CHROME_PROFILE)):
        raise FileNotFoundError(
            f"Profile {CHROME_PROFILE!r} not found under {CHROME_BASE_DIR}. "
        )

    if os.path.exists(CHROME_PROFILE_CLONE_DIR):
        shutil.rmtree(CHROME_PROFILE_CLONE_DIR)#limpa se ja tiver um no lugar que queremos clonar
    shutil.copytree(CHROME_BASE_DIR, CHROME_PROFILE_CLONE_DIR, ignore=DONT_COPY_THIS_WHEN_CLONNING_GOOGLE)
    return CHROME_PROFILE_CLONE_DIR


def create_stealth_driver(user_data_dir):
    import undetected_chromedriver as uc

    if not CHROME_BINARY:
        raise RuntimeError("No Chrome/Chromium binary found on this machine")

    options = Options()
    options.binary_location = str(CHROME_BINARY)
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-extensions')
    options.add_argument('--disable-plugins')
    options.add_argument(f'--profile-directory={CHROME_PROFILE}')
    options.set_capability('goog:loggingPrefs', {'browser': 'ALL', 'performance': 'ALL'})

    print(f"Launching Chrome (version_main={PROCESSED_VERSION}, profile={user_data_dir})")
    driver = uc.Chrome(
        options=options,
        browser_executable_path=str(CHROME_BINARY),
        version_main=PROCESSED_VERSION,
        user_data_dir=user_data_dir,
    )
    try:
        driver.execute_cdp_cmd('Network.enable', {})
        driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {'source': STEALTH_JS})
    except Exception as exc:
        print(f"CDP setup partially failed: {exc}")
    print("Driver ready.")
    return driver


## 3. Volza table helpers
Expanding a row reveals a `table.more-table` with all 31 fields as label/value pairs - that is the scrape target, not the wide collapsed table.

Two gotchas the code works around: the page renders two `.ant-table-body` tables and only one is inside the viewport (the other sits at `left ~= -1146`, where expanding works but is invisible), and rows must be expanded one click per pass because React re-renders after each one.

In [42]:
VOLZA_HOME = "https://app.volza.com/home"
LOGIN_MARKER = "ob-pending-card-title"
EXPAND_SELECTOR = "button.ant-table-row-expand-icon.ant-table-row-expand-icon-collapsed"


In [ ]:
def session():
    clone_path = clone_chrome_profile()
    driver = create_stealth_driver(clone_path)
    driver.set_page_load_timeout(30)
    driver.get(VOLZA_HOME)
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.CLASS_NAME, LOGIN_MARKER))
    )
    print(f"Logged-in session confirmed ({LOGIN_MARKER} present)")
    return driver


def on_screen_table(driver):
    infos = driver.execute_script(
        """
        var out = [];
        document.querySelectorAll('.ant-table-body table').forEach(function(t, i) {
            var r = t.getBoundingClientRect();
            out.push({index: i, onScreen: (r.left >= 0 && r.left < window.innerWidth)});
        });
        return out;
        """
    )
    tables = driver.find_elements(By.CSS_SELECTOR, ".ant-table-body table")
    for info in infos:
        if info['onScreen'] and info['index'] < len(tables):
            return tables[info['index']]
    return tables[0] if tables else None


def first_row_key(driver):
    table = on_screen_table(driver)
    if table is None:
        return None
    return driver.execute_script(
        "var r = arguments[0].querySelector('tr[data-row-key]');"
        "return r ? r.getAttribute('data-row-key') : null;",
        table,
    )


def goto_workspace(driver, url):
    print(f"  navigating to {url}")
    driver.get(url)
    try:
        WebDriverWait(driver, 40).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".ant-table-body table"))
        )
    except TimeoutException:
        print("  no .ant-table-body table appeared within timeout")
        return False
    human_time_sleep(2)
    table = on_screen_table(driver)
    n_expand = len(table.find_elements(By.CSS_SELECTOR, EXPAND_SELECTOR)) if table is not None else 0
    print(f"  table ready; first row key={first_row_key(driver)}, collapsed expand buttons={n_expand}")
    return True


def expand_all_rows(driver):
    max_clicks = 300
    settle = 0.15
    clicked = 0
    while clicked < max_clicks:
        table = on_screen_table(driver)
        if table is None:
            break
        try:
            buttons = table.find_elements(By.CSS_SELECTOR, EXPAND_SELECTOR)
        except StaleElementReferenceException:
            continue
        if not buttons:
            break
        before = len(buttons)
        try:
            driver.execute_script("arguments[0].click();", buttons[0])
        except StaleElementReferenceException:
            continue
        clicked += 1
        human_time_sleep(settle)
        table = on_screen_table(driver)
        after = len(table.find_elements(By.CSS_SELECTOR, EXPAND_SELECTOR)) if table is not None else 0
        if after >= before:
            print(f"    click {clicked} left the collapsed count at {before} -> {after}; stopping early")
            break
    return clicked


def scrape_expanded(driver):
    table = on_screen_table(driver)
    if table is None:
        return []
    return driver.execute_script(
        r"""
        var out = [];
        arguments[0].querySelectorAll('tr.ant-table-expanded-row').forEach(function(tr) {
            var rec = {};
            var prev = tr.previousElementSibling;
            rec['_row_key'] = prev ? prev.getAttribute('data-row-key') : null;
            tr.querySelectorAll('table.more-table tr').forEach(function(r) {
                var tds = r.querySelectorAll('td');
                if (tds.length >= 2) {
                    var key = (tds[0].innerText || '').trim().replace(/:\s*$/, '');
                    if (key) rec[key] = (tds[1].innerText || '').trim();
                }
            });
            out.push(rec);
        });
        return out;
        """,
        table,
    )


def click_pagination(driver, xpath, li_index):
    timeout = 25
    before_key = first_row_key(driver)
    target = None
    try:
        target = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, xpath))
        )
        print(f"    found via absolute XPath: text={target.text!r}")
    except (TimeoutException, NoSuchElementException):
        items = driver.find_elements(By.CSS_SELECTOR, "ul.ant-pagination > li")
        print(f"    absolute XPath failed; ul.ant-pagination has {len(items)} <li>")
        if len(items) >= li_index:
            target = items[li_index - 1]
            print(f"    falling back to li[{li_index}]: text={target.text!r}")
    if target is None:
        print("    no pagination element found")
        return False

    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", target)
    human_time_sleep(0.3)
    driver.execute_script("arguments[0].click();", target)

    deadline = time.time() + timeout
    while time.time() < deadline:
        human_time_sleep(0.5)
        now_key = first_row_key(driver)
        if now_key and now_key != before_key:
            print(f"    page changed: first row key {before_key} -> {now_key}")
            return True
    print(f"    first row key never changed from {before_key} within {timeout}s")
    return False


def scrape_workspace(driver, url, page_targets=(), label_prefix="page"):
    if not goto_workspace(driver, url):
        return []

    records = []
    steps = [(f"{label_prefix}_1", None, None)]
    steps += [(f"{label_prefix}_li{li}", xpath, li) for xpath, li in page_targets]

    for label, xpath, li_index in steps:
        print(f"\n=== {label} ===")
        if xpath is not None:
            if not click_pagination(driver, xpath, li_index):
                print(f"  skipping {label}: could not navigate")
                continue
            human_time_sleep(2)

        n_clicked = expand_all_rows(driver)
        page_records = scrape_expanded(driver)
        for rec in page_records:
            rec['_page'] = label
            rec['_source_url'] = url
        records.extend(page_records)
        print(f"  expanded {n_clicked} rows, scraped {len(page_records)} records")
    return records


def report_and_save(records, out_csv):
    df = pd.DataFrame(records)
    print(f"\nDataFrame shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    if not df.empty:
        print(f"Records per page:\n{df['_page'].value_counts()}")
        print(f"Duplicate _row_key values: {df['_row_key'].duplicated().sum()}")
        print("\nFirst rows:")
        print(df.head(3).to_string())
    df.to_csv(out_csv, index=False)
    print(f"\nSaved {len(df)} rows to {out_csv}")
    return df


## 3b. ImportYeti supplier table


In [44]:
YETI_TBODY_XPATH = "/html/body/div[2]/main/div/div/div[6]/div[2]/section/div[2]/div[1]/table/tbody"
EXPAND_SUPPLYERS_XPATH = "/html/body/div[2]/main/div/div/div[6]/div[2]/section/div[2]/div[2]/div/span"
SHIPMENTS_COL = "shipments (01/2015 - 08/2026)"


In [45]:
def yeti_row_count(driver):
    return driver.execute_script(
        "var t = document.evaluate(arguments[0], document, null, 9, null).singleNodeValue;"
        "return t ? t.querySelectorAll(':scope > tr').length : 0;",
        YETI_TBODY_XPATH,
    )


def expand_yeti_suppliers(driver):
    before = yeti_row_count(driver)
    try:
        button = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, EXPAND_SUPPLYERS_XPATH))
        )
    except TimeoutException:
        print(f"  no expand control found; rows stay at {before}")
        return before

    print(f"  expand control text={button.text[:80]!r}, rows before={before}")
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", button)
    human_time_sleep(0.3)
    driver.execute_script("arguments[0].click();", button)

    grown = before
    deadline = time.time() + 30
    while time.time() < deadline:
        human_time_sleep(0.5)
        now = yeti_row_count(driver)
        if now > grown:
            grown = now
            continue
        if grown > before:
            print(f"  expanded: rows {before} -> {grown}")
            return grown
    print(f"  row count never grew from {before} within 30s")
    return before


def scrape_yeti_table(driver):
    tbody = WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.XPATH, YETI_TBODY_XPATH))
    )
    rows = driver.execute_script(
        """
        var out = [];
        arguments[0].querySelectorAll(':scope > tr').forEach(function(tr) {
            var tds = tr.querySelectorAll(':scope > td');
            if (!tds.length) return;
            var anchors = tds[0].querySelectorAll(':scope > div > div:nth-of-type(1) > a');
            var texts = Array.prototype.map.call(anchors, function(a) {
                return (a.innerText || '').trim();
            });
            var city = texts[1] || '';
            var country = texts[2] || '';
            if (!country) {
                country = city;
                city = '';
            }
            var span = tds.length >= 3
                ? tds[2].querySelector(':scope > div:nth-of-type(1) > span')
                : null;
            out.push({
                name: texts[0] || '',
                city: city,
                country: country,
                shipments: span ? (span.innerText || '').trim() : '',
                n_anchors: texts.length
            });
        });
        return out;
        """,
        tbody,
    )
    kept = [r for r in rows if r['name']]
    print(f"  tbody rows={len(rows)}, with name={len(kept)}")
    return kept


def save_yeti(rows, out_csv):
    df = pd.DataFrame(rows)
    if df.empty:
        print("no rows scraped")
        return df
    print(f"\nanchors per row:\n{df['n_anchors'].value_counts().sort_index()}")
    print(f"missing city={(df['city'] == '').sum()} "
          f"country={(df['country'] == '').sum()} "
          f"shipments={(df['shipments'] == '').sum()}")
    df = df.drop(columns='n_anchors').rename(columns={'shipments': SHIPMENTS_COL})
    df = df[['name', 'city', 'country', SHIPMENTS_COL]]
    print(f"\n{df.head(10).to_string()}")
    df.to_csv(out_csv, index=False)
    print(f"\nSaved {len(df)} rows to {out_csv}")
    return df

## 4. Run

In [46]:
"""driver = session()"""

'driver = session()'

In [47]:
# Workspace 24846649 / Shipments: initial page plus the two pagination steps.
URL_SHIPMENTS = "https://app.volza.com/workspace/search/24846649#Shipments"

# Absolute XPaths for the two pagination targets, each with its <li> index as
# the fallback used when the XPath stops resolving.
SHIPMENT_PAGE_TARGETS = [
    ("/html/body/div[1]/div/div/div[3]/article/div/div/div/div/div[2]/div/div/div/div[2]/div/div[2]"
     "/div[2]/div/div[2]/div[2]/div[2]/div/ul/li[4]", 4),
    ("/html/body/div[1]/div/div/div[3]/article/div/div/div/div/div[2]/div/div/div/div[2]/div/div[2]"
     "/div[2]/div/div[2]/div[2]/div[2]/div/ul/li[5]", 5),
]
VOLZA_CSV = "volza_shipments.csv"

In [48]:
"""

df_shipments = report_and_save(
    scrape_workspace(driver, URL_SHIPMENTS, SHIPMENT_PAGE_TARGETS),
    os.path.join(os.getcwd(), VOLZA_CSV),
)"""

'\n\ndf_shipments = report_and_save(\n    scrape_workspace(driver, URL_SHIPMENTS, SHIPMENT_PAGE_TARGETS),\n    os.path.join(os.getcwd(), VOLZA_CSV),\n)'

## 4b. Volza: porto de descarga -> distrito provavel


In [49]:
PORTO_DISTRITO = {
    "Charleston": "Charleston, SC",
    "New York Newark Area Newark New Jersey": "New York, NY",
    "Houston": "Houston-Galveston, TX",
    "Los Angeles": "Los Angeles, CA",
    "Norfolk": "Norfolk, VA",
    "New York": "New York, NY",
    "Savannah": "Savannah, GA",
    "Miami": "Miami, FL",
    "Tacoma": "Seattle, WA",
    "Philadelphia": "Philadelphia, PA",
}

# Distrito PROVAVEL, nao certo: "Port of Destination" e o porto de descarga do
# navio, e a carga pode seguir in-bond e ser desembaraçada no interior. Nas rotas
# oceanicas isso vale de 28% a 69% do peso, entao a coluna e hipotese, nao chave.
VOLZA_CSV = os.path.join(os.getcwd(), "volza_shipments.csv")


In [50]:
"""df_volza = pd.read_csv(VOLZA_CSV)
df_volza["district_provavel"] = df_volza["Port of Destination"].map(PORTO_DISTRITO)
df_volza.to_csv(VOLZA_CSV, index=False)

print(f"{len(df_volza)} linhas, {len(df_volza.columns)} colunas")
print(df_volza["district_provavel"].value_counts(dropna=False).to_string())"""

'df_volza = pd.read_csv(VOLZA_CSV)\ndf_volza["district_provavel"] = df_volza["Port of Destination"].map(PORTO_DISTRITO)\ndf_volza.to_csv(VOLZA_CSV, index=False)\n\nprint(f"{len(df_volza)} linhas, {len(df_volza.columns)} colunas")\nprint(df_volza["district_provavel"].value_counts(dropna=False).to_string())'

In [51]:
def volza_solo(path):
    "(distrito, mes) com exatamente 1 embarque no Volza. Como a celula tem um unico embarque, o peso dela e atribuivel a um shipper nomeado."
    v = pd.read_csv(path)
    v["date"] = pd.to_datetime(v["Date"], errors="coerce")
    v["mes"] = v["date"].dt.to_period("M").dt.to_timestamp()
    v["kg"] = pd.to_numeric(v["Gross Weight"], errors="coerce")
    v["hs6"] = v["HS Code"].astype(str).str[:6]

    v = v.dropna(subset=["district_provavel", "mes"])
    n = v.groupby(["district_provavel", "mes"])["kg"].transform("size")
    solo = v[n == 1]
    return solo[["district_provavel", "mes", "hs6", "Country of Origin",
                 "Shipper", "Consignee", "kg"]].sort_values(["district_provavel", "mes"])


df_solo = volza_solo(VOLZA_CSV)
todas = pd.read_csv(VOLZA_CSV).dropna(subset=["district_provavel"])

print(f"celulas (distrito, mes) unicas: "
      f"{todas.groupby(['district_provavel', pd.to_datetime(todas['Date']).dt.to_period('M')]).ngroups}")
print(f"com exatamente 1 embarque: {len(df_solo)}\n")
print(df_solo.to_string(index=False))

celulas (distrito, mes) unicas: 22
com exatamente 1 embarque: 13

    district_provavel        mes    hs6 Country of Origin                           Shipper                               Consignee     kg
       Charleston, SC 2026-02-01 848210            Brazil                SKF DO Brasil Ltda                            SKF USA Inc.    556
Houston-Galveston, TX 2025-12-01 730630             India                 Z To Order and NA                       Z To Order and NA 184253
Houston-Galveston, TX 2026-01-01 848210            Mexico  Liebherr Monterrey S de RL de CV       HIDROMEK HIDROLIK VE MEKANIK MAKI  18480
      Los Angeles, CA 2025-11-01 848210             India   CTL Logistics (India) Pvt. Ltd.                            CTLLAX, Inc.  19500
      Los Angeles, CA 2026-01-01 848210            Mexico                 Z To Order and NA REILO LOGISTICA ADUANAL S DE R L DE C V   3949
            Miami, FL 2025-11-01 848210            Mexico                Liebherr Monterrey      Red

## 4c. Quem nao e fornecedor

Duas exclusoes, so isso. **Transportadora**: agente de carga que aparece como shipper
mas nao fabrica nada. **So matriz**: exportador cujos embarques vao todos para a
propria marca nos EUA — logistica interna de um concorrente, nao fornecedor
alternativo.

A regra da segunda e marca repetida nos dois lados do embarque. Tirando forma
juridica, pais e palavra de setor da razao social, o que sobra e a marca; se ela
aparece no shipper e no consignee, o embarque e intragrupo.

Sem janela de tempo: cativo e "nunca apareceu vendendo para fora", entao cortar por
data inventaria cativo que so nao foi observado o bastante. Vale o arquivo inteiro do
Volza — que e a unica fonte com destinatario, e por isso a unica que consegue
responder isso.

O resultado nao fica so na tela: `excluir()` alimenta o scraping da secao 5, entao
transportadora e cativo saem antes de virar linha de CSV.

In [68]:
# Forma juridica, pais e palavra de setor nao identificam ninguem: duas empresas
# sem relacao compartilham "Ltda", "Brasil" ou "Bearings". O que sobra depois de
# remove-las e a marca — e marca repetida nos dois lados do embarque significa
# que o exportador esta vendendo para a propria matriz.
GENERICO = set("""
ltd ltda limited inc llc llp corp corporation company companhia cia sa saa gmbh
plc pvt private srl bv nv kg oy ab spa sas cv rl group holding holdings jsc
usa america american brasil brazil brazilian india indian mexico mexicana
mexicano canada canadian china int intl international global world north south
new the and for
logistics logistica logistic logistik transport transportes transportation
transp freight forwarding shipping cargo carrier carriers spedition agenciamento
bearing bearings tube tubes tubular tubulars steel metal metals metalurgica
industria industries industrial engineering manufacturing manufactura technology
technologies parts distribution solutions services servicos serv comercio
comercial comercializadora exportadora importadora trading supply sourcing
systems machinery equipamentos
""".split())

# Nome que nao e nome: o Volza copia o texto do BL quando ele diz "to order".
PLACEHOLDER = r"^(z )?to order|not available|^n/?a$|unknown"

# 3PL e agente de carga aparecem como shipper mas nao fabricam nada. Os nomes
# proprios entram porque sao os operadores que dominam a rota e a regex generica
# nao os pega ("Kuehne Nagel Serv Logist" nao contem "logistic").
FORWARDER = (r"logisti|forward|freight|carrier|transp|agenciam|spedition|"
             r"to order|princess|cruise|schenker|panalpina|kuehne|damco|"
             r"expeditors|dsv|dhl|allink|maersk")

In [69]:
def marca(nome):
    """Tokens distintivos de uma razao social: tira forma juridica, pais e palavra
    de setor, e o que sobra e a marca."""
    return {w for w in re.findall(r"[a-z]+", str(nome).lower())
            if len(w) >= 3 and w not in GENERICO}


def intragrupo(shipper, consignee):
    """Marca repetida nos dois lados do embarque = exportador embarcando para a
    propria matriz. Placeholder de BL nao conta: "to order" nao e nome."""
    if re.search(PLACEHOLDER, str(consignee).lower()):
        return False
    return bool(marca(shipper) & marca(consignee))


v = pd.read_csv(VOLZA_CSV)
v = v[[not re.search(PLACEHOLDER, str(s).lower()) for s in v["Shipper"]]]
v["intra"] = [intragrupo(s, c) for s, c in zip(v["Shipper"], v["Consignee"])]

# Todo o arquivo, sem janela: cativo e "nunca apareceu vendendo para fora", entao
# cortar por data so inventaria cativo que na verdade nao foi observado o bastante.
p = v.groupby(["Country of Origin", "Shipper"]).agg(
    n_embarques=("intra", "size"), n_intragrupo=("intra", "sum")).reset_index()
p["transportadora"] = [bool(re.search(FORWARDER, s.lower())) for s in p["Shipper"]]
p["so_matriz"] = ~p["transportadora"] & (p["n_intragrupo"] == p["n_embarques"])

COLS = ["Country of Origin", "Shipper", "n_embarques", "n_intragrupo"]
print(f"Volza: {len(v)} embarques nomeados, {len(p)} exportadores, "
      f"{v['Date'].min()} a {v['Date'].max()}")
print(f"\nTRANSPORTADORAS ({int(p['transportadora'].sum())}) — agente de carga no "
      f"lugar do fabricante")
print(p[p["transportadora"]][COLS].to_string(index=False))
print(f"\nSO EMBARCAM PARA A PROPRIA MATRIZ ({int(p['so_matriz'].sum())}) — todos os "
      f"embarques intragrupo")
print(p[p["so_matriz"]].sort_values("n_embarques", ascending=False)[COLS]
      .to_string(index=False))


# Marcas que saem do scraping do ImportYeti. Cativo so o Volza sabe dizer, porque e
# a unica fonte com destinatario; transportadora sai pelo proprio nome e nao precisa
# de fonte nenhuma.
CATIVOS = frozenset().union(frozenset(),
                            *[marca(s) for s in p.loc[p["so_matriz"], "Shipper"]])


def excluir(nome):
    """Roda sobre o nome cru do ImportYeti, que nao traz destinatario: o veredito do
    Volza e estendido por marca. Uma marca pega com um unico embarque cativo num pais
    passa a excluir a mesma marca em todos — e o preco de nao ter destinatario la."""
    return bool(re.search(FORWARDER, str(nome).lower()) or marca(nome) & CATIVOS)


print(f"\nmarcas cativas herdadas pelo ImportYeti: {', '.join(sorted(CATIVOS))}")

grupos cativos no Volza: deere, john, nsk, schaeffler, skf
shippers mistos (matriz + terceiro): (nenhum)

730630 — capacidade (Volza, lote tipico, n >= 3, ordenado por acesso e kg para terceiro)
Country of Origin                              Shipper   acesso  share_aberto  n_embarques  n_aberto  kg_mediano  kg_mediano_aberto  kg_max
            India Metamorphosis Engitech India Pvt Ltd   aberto           1.0            3         3     63790.0            63790.0   63930
            India                    Z To Order and NA sem nome           0.0            6         0    111330.0                NaN  184253

730630 — dinamica (ImportYeti, top 12 por escala recente)
                                     liveness_meses  escala_12m  tendencia  resiliencia          acesso
name                                                                                                   
Goodluck Industries                             0.0       37.75       1.07         8.93  nao verificado
Tube Investmen

## 5. ImportYeti scraping


In [52]:
PROBE_BASES = {
    "730630": "https://www.importyeti.com/hs-codes/730630-other-welded-of-circular-cross-section-of",
    "848210": "https://www.importyeti.com/hs-codes/848210-ball-bearings"}
PROBE_COUNTRIES = ("brazil", "canada", "mexico", "india")
PROBE_TARGETS = [(hts, c, f"{base}/{c}")
                 for hts, base in PROBE_BASES.items() for c in PROBE_COUNTRIES]
PROBE_SUPPLYER_XPATH = "/html/body/div[2]/main/div/div/div[6]/div[2]/div/div[2]/div/div"
PROBE_SVG_CSS = "table tbody tr td svg.recharts-surface"
PROBE_BAR_CSS = ".recharts-bar-rectangle path"
HOVER_NUDGES = (0, 1, -1, 2, -2, 3, -3)
HOVER_MOVE_MS = 20
TOTAL_ID = "total_shipments_over_time"
TOTAL_SVG_CSS = f"#{TOTAL_ID} svg.recharts-surface"
NAME_XPATH = "ancestor::tr[1]/td[1]/div/div[1]/a[1]"


In [53]:
"""def find(by, value, timeout=30):
    "A pagina monta as secoes so quando entram na viewport, entao esperar pela presenca sem rolar estoura timeout. Rola de tela em tela ate o no existir."
    deadline = time.time() + timeout
    while time.time() < deadline:
        found = driver.find_elements(by, value)
        if found:
            return found[0]
        driver.execute_script(
            "var y = window.scrollY;"
            "window.scrollBy(0, window.innerHeight * 0.8);"
            "if (window.scrollY === y) window.scrollTo(0, 0);")   # chegou ao fim: recomeca
        human_time_sleep(0.4)
    raise TimeoutException(f"{value} nao apareceu apos rolar a pagina inteira")


def probe_read(svg, bar, single):
    parent = svg.find_element(By.XPATH, "..")
    tips = parent.find_elements(By.CSS_SELECTOR, ".recharts-tooltip-wrapper")
    if len(tips) == 0:
        return None
    fields = [p.get_attribute("textContent").strip()
              for p in tips[0].find_elements(By.CSS_SELECTOR, "p")]

    cursors = svg.find_elements(By.CSS_SELECTOR, ".recharts-tooltip-cursor")
    if len(cursors) == 0:
        if single and fields and tips[0].is_displayed():
            return fields[:2]
        return None
    cursor_x = float(cursors[0].get_attribute("x"))
    cursor_w = float(cursors[0].get_attribute("width"))

    bar_mid = float(bar.get_attribute("x")) + float(bar.get_attribute("width")) / 2
    if cursor_x <= bar_mid <= cursor_x + cursor_w:
        return fields[:2]
    return None


def hover_bar(svg, bar, single):
    dx0 = driver.execute_script(
        "var s = arguments[0].getBoundingClientRect(), b = arguments[1].getBoundingClientRect();"
        "return b.left + b.width / 2 - (s.left + s.width / 2);",
        svg, bar,
    )
    for dx in HOVER_NUDGES:
        ActionChains(driver, duration=HOVER_MOVE_MS).move_to_element_with_offset(
            svg, int(round(dx0 + dx)), 0).perform()
        human_time_sleep(0.001)
        fields = probe_read(svg, bar, single)
        if fields:
            return fields
    return []


def field_value(text):
    return text.split(":", 1)[1].strip() if ":" in text else text


def scrape_bars(svg, key_name, key_value):
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", svg)
    bars = svg.find_elements(By.CSS_SELECTOR, PROBE_BAR_CSS)
    out = []
    for bar in bars:
        fields = hover_bar(svg, bar, len(bars) == 1)
        if fields:
            out.append({key_name: key_value,
                        "date": field_value(fields[0]),
                        "shipments": field_value(fields[1])})
    return out


country_rows = []
supplier_chart_rows = []
supplier_table_rows = []

for hts, country, url in PROBE_TARGETS:
    print(f"\n=== {hts} / {country} ===")
    driver.get(url)

    container = find(By.ID, TOTAL_ID)
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", container)
    country_svg = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, TOTAL_SVG_CSS))
    )
    country_rows += [{**r, "hts": hts}
                     for r in scrape_bars(country_svg, "country", country)]
    print(f"  country chart: {len(country_rows)} rows so far")

    target = find(By.XPATH, PROBE_SUPPLYER_XPATH)
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", target)
    human_time_sleep(0.5)
    driver.execute_script("arguments[0].click();", target)
    human_time_sleep(2)
    expand_yeti_suppliers(driver)

    supplier_table_rows += [{**r, "hts": hts} for r in scrape_yeti_table(driver)
                            if not excluir(r["name"])]

    charts = driver.find_elements(By.CSS_SELECTOR, PROBE_SVG_CSS)
    print(f"  supplier charts: {len(charts)}")
    for svg in charts:
        names = svg.find_elements(By.XPATH, NAME_XPATH)
        name = names[0].get_attribute("textContent").strip() if names else ""
        if excluir(name):          # transportadora ou cativo: nao vale hover
            continue
        supplier_chart_rows += [{**r, "hts": hts, "country": country}
                                for r in scrape_bars(svg, "name", name)]
    print(f"  supplier chart rows so far: {len(supplier_chart_rows)}")

df_country = pd.DataFrame(country_rows)
df_supplier_charts = pd.DataFrame(supplier_chart_rows)

for hts in PROBE_BASES:
    out = lambda kind: os.path.join(os.getcwd(), f"importyeti_{kind}_{hts}.csv")
    df_country[df_country.hts == hts].drop(columns="hts").to_csv(out("country_shipments"), index=False)
    df_supplier_charts[df_supplier_charts.hts == hts].drop(columns="hts").to_csv(out("supplier_shipments"), index=False)
    save_yeti([r for r in supplier_table_rows if r["hts"] == hts], out("suppliers"))"""

'def find(by, value, timeout=30):\n    "A pagina monta as secoes so quando entram na viewport, entao esperar pela presenca sem rolar estoura timeout. Rola de tela em tela ate o no existir."\n    deadline = time.time() + timeout\n    while time.time() < deadline:\n        found = driver.find_elements(by, value)\n        if found:\n            return found[0]\n        driver.execute_script(\n            "var y = window.scrollY;"\n            "window.scrollBy(0, window.innerHeight * 0.8);"\n            "if (window.scrollY === y) window.scrollTo(0, 0);")   # chegou ao fim: recomeca\n        human_time_sleep(0.4)\n    raise TimeoutException(f"{value} nao apareceu apos rolar a pagina inteira")\n\n\ndef probe_read(svg, bar, single):\n    parent = svg.find_element(By.XPATH, "..")\n    tips = parent.find_elements(By.CSS_SELECTOR, ".recharts-tooltip-wrapper")\n    if len(tips) == 0:\n        return None\n    fields = [p.get_attribute("textContent").strip()\n              for p in tips[0].find_el

## 5b. Ranking de fornecedores

Tres metricas, todas da serie trimestral do ImportYeti, ja sem transportadora e sem
cativo. `liveness` e portao, nao nota: quem parou de embarcar nao entra. `score` e
escala vezes tendencia — embarques por trimestre projetados, uma quantidade, nao um
indice com pesos escolhidos a mao.

Os melhores de cada pais passam primeiro, depois disputam misturados. E essa a ordem
que evita o defeito do ranking antigo, que comparava fornecedores de paises diferentes
por caracteristica da rota em vez de do fornecedor.

In [ ]:
TOPO_PAIS = 3          # melhores de cada pais que passam para a mistura
MAX_LIVENESS = 2       # trimestres sem embarcar antes de sair do ranking


def ranquear(hts):
    """Escala e tendencia em trimestres, porque a barra do ImportYeti e trimestral.
    `score` = escala x tendencia = embarques por trimestre projetados, nao um indice
    com pesos inventados. Liveness e portao, nao nota: quem parou nao ranqueia."""
    s = pd.read_csv(os.path.join(os.getcwd(), f"importyeti_supplier_shipments_{hts}.csv"))
    s = s[[not excluir(n) for n in s["name"]]]
    s["tri"] = pd.to_datetime(s["date"], format="%b %Y")

    tri = sorted(s["tri"].unique())
    m = (s.pivot_table(index=["country", "name"], columns="tri", values="shipments",
                       aggfunc="sum").reindex(columns=tri).fillna(0))

    r = pd.DataFrame(index=m.index)
    r["escala"] = m[tri[-4:]].sum(axis=1) / 4
    r["tendencia"] = r["escala"] * 4 / m[tri[-8:-4]].sum(axis=1).replace(0, np.nan)
    r["liveness_tri"] = len(tri) - 1 - m.apply(
        lambda linha: np.flatnonzero(linha.values > 0).max(), axis=1)
    r["score"] = r["escala"] * r["tendencia"].fillna(1)

    vivos = r[r["liveness_tri"] <= MAX_LIVENESS].sort_values("score", ascending=False)
    return vivos.groupby(level="country").head(TOPO_PAIS).sort_values(
        "score", ascending=False)


RANKING_CSV = os.path.join(os.getcwd(), "suppliers_ranked.csv")

ranking = []
for hts in ["730630", "848210"]:
    top = ranquear(hts)
    ranking.append(top.reset_index().assign(hts=hts))
    print(f"\n{'='*72}\n{hts} — top {TOPO_PAIS} de cada pais, ranqueados misturados")
    print(top.round(2).to_string())

df_ranking = pd.concat(ranking)[["hts", "country", "name", "escala", "tendencia",
                                 "liveness_tri", "score"]]
df_ranking.to_csv(RANKING_CSV, index=False)
print(f"\nSalvo {len(df_ranking)} fornecedores em {RANKING_CSV}")

## 6. USITC DataWeb

`runReport` nao aceita o schema achatado do `/v3/api-docs`; o corpo real e aninhado (`reportOptions` / `searchOptions` / `sortingAndDataFormat`), obtido de `getAllSystemSavedQueries`. A validacao le `countriesExpanded`, nao `countries`. Cada relatorio tem teto de 20.000 linhas, por isso a consulta e fatiada por ano.


In [54]:
DATAWEB_URL = "https://datawebws.usitc.gov/dataweb/api/v2/report2/runReport"
DATAWEB_MEASURES = ["CONS_COST_INS_FREIGHT+CONS_CALC_DUTY", "CONS_CUSTOMS_VALUE",
                    "CONS_CHARGES_INS_FREIGHT", "CONS_CALC_DUTY",
                    "CONS_FIR_UNIT_QUANT", "CONS_SEC_UNIT_QUANT"]
DATAWEB_COUNTRIES = [("2010", "Mexico - MX - MEX"), ("5330", "India - IN - IND"),
                     ("1220", "Canada - CA - CAN"), ("3510", "Brazil - BR - BRA")]
DATAWEB_COMMODITIES = ["730630", "848210"]


In [55]:
def dataweb_token():
    token = os.environ.get("DATAWEB_TOKEN")
    if token:
        return token.strip()
    path = os.path.expanduser("~/.dataweb_token")
    if os.path.exists(path):
        return open(path).read().strip()
    raise RuntimeError("set DATAWEB_TOKEN or create ~/.dataweb_token")


def dataweb_payload(start_date, end_date):
    return {
        "savedQueryName": "", "savedQueryDesc": "", "isOwner": True, "runMonthly": True,
        "apiToken": "", "captchaResponse": "", "captchaValid": False, "queryJSON": "",
        "deletedCountryUserGroups": [], "deletedCommodityUserGroups": [],
        "deletedDistrictUserGroups": [],
        "expandedGroups": {"commodities": [], "countries": [], "districts": []},
        "reportOptions": {"tradeType": "Import", "classificationSystem": "HTS"},
        "searchOptions": {
            "componentSettings": {
                "dataToReport": DATAWEB_MEASURES, "scale": "1",
                "timeframeSelectType": "specificDateRange", "years": [],
                "startDate": start_date, "endDate": end_date,
                "startMonth": None, "endMonth": None, "yearsTimeline": "Monthly"},
            "commodities": {
                "commodities": DATAWEB_COMMODITIES,
                "commoditiesExpanded": [{"text": c, "value": c, "name": c}
                                        for c in DATAWEB_COMMODITIES],
                "commoditiesManual": None,
                "commodityGroups": {"systemGroups": [], "userGroups": []},
                "granularity": "6", "searchGranularity": "6", "groupGranularity": "6",
                "aggregation": "Break Out Commodities", "codeDisplayFormat": "NO",
                "commoditySelectType": "list", "showHTSValidDetails": True},
            "countries": {
                "countries": [v for v, _ in DATAWEB_COUNTRIES],
                "countriesExpanded": [{"text": n, "value": v, "name": n}
                                      for v, n in DATAWEB_COUNTRIES],
                "countryGroups": {"systemGroups": [], "userGroups": []},
                "aggregation": "Break Out Countries", "countriesSelectType": "list"},
            "MiscGroup": {
                "importPrograms": {"importPrograms": [], "aggregation": "Aggregate CSC"},
                "extImportPrograms": {"programsSelectType": "all", "extImportPrograms": [],
                                      "extImportProgramsExpanded": [],
                                      "aggregation": "Aggregate CSC"},
                "provisionCodes": {"rateProvisionCodes": [], "rateProvisionCodesExpanded": [],
                                   "aggregation": "Aggregate RPCODE",
                                   "provisionCodesSelectType": "all",
                                   "rateProvisionGroups": {"systemGroups": []}},
                "districts": {"districts": [], "districtsExpanded": [],
                              "districtGroups": {"userGroups": []},
                              "aggregation": "Break Out District",
                              "districtsSelectType": "all"}}},
        "sortingAndDataFormat": {
            "DataSort": {"sortOrder": [], "columnOrder": ["null"], "sortYear": None},
            "reportCustomizations": {"totalRecords": "20000", "exportCombineTables": False,
                                     "reportsGrid": True, "removeDuplicateValues": False,
                                     "suppressZeroValues": False, "displayCommodityList": False,
                                     "reportsFontSize": "m", "exportRawData": True}},
        "unitConversion": "0", "manualConversions": []}


def dataweb_run(start_date, end_date):
    body = json.dumps(dataweb_payload(start_date, end_date)).encode()
    headers = {"Authorization": "Bearer " + dataweb_token(),
               "Content-Type": "application/json"}
    delay = 30
    for attemp in range(7):
        try:
            request = urllib.request.Request(DATAWEB_URL, data=body,
                                             method="POST", headers=headers)
            print(f"running attemp: {attemp}")
            with urllib.request.urlopen(request, timeout=30) as response:
                return json.loads(response.read().decode())["dto"]
        except Exception as exc:
            print(f"{type(exc).__name__}: {exc}, will retry again")
            time.sleep(delay)
            delay = min(delay * 2, 300)
    return None


def dataweb_parse(dto):
    rows = []
    for table in dto.get("tables") or []:
        measure = table["tableInfo"]["dataToReportDesc"]
        groups = table["column_groups"]
        keys = [c["label"] for c in groups[0]["columns"]]
        months = [c["label"] for c in groups[1]["columns"]]
        unit = groups[1].get("label")
        for group in table.get("row_groups") or []:
            for row in group.get("rowsNew") or []:
                cells = [e["value"] for e in row["rowEntries"]]
                base = dict(zip(keys, cells[:len(keys)]))
                for month, value in zip(months, cells[len(keys):]):
                    rows.append({**base, "Measure": measure, "Unit": unit,
                                 "Month": month, "Value": value})
    return rows


def dataweb_year(year, end_year_month):
    dto = dataweb_run(f"01/{year}", end_year_month)
    if dto is None:
        print(f"{year}: no data returned")
        return []
    if dto.get("errors"):
        print(f"{year}: {dto['errors']}")
        return []

    rows = dataweb_parse(dto)
    sizes = [t.get("totalRowSize") for t in dto.get("tables") or []]
    print(f"{year}: {len(rows):6} rows  tableRowSizes={sizes}")
    human_time_sleep(10)
    return rows


In [56]:
DATAWEB_START_YEAR = 2022
DATAWEB_END_YEAR = 2026
DATAWEB_END_MONTH = "06/2026"


In [57]:
dataweb_rows = []
for year in range(DATAWEB_START_YEAR, DATAWEB_END_YEAR + 1):
    end = DATAWEB_END_MONTH if year == DATAWEB_END_YEAR else f"12/{year}"
    dataweb_rows += dataweb_year(year, end)

df_dataweb = pd.DataFrame(dataweb_rows)
df_dataweb.to_csv(os.path.join(os.getcwd(), "debug_before_cleaning.csv"), index=False)

print(f"linhas long: {len(df_dataweb)}")
print(df_dataweb.dtypes.to_string())
print(df_dataweb["Measure"].unique())


running attemp: 0
2022:   8244 rows  tableRowSizes=[123, 123, 123, 123, 123, 72]
running attemp: 0
2023:   7800 rows  tableRowSizes=[116, 116, 116, 116, 116, 70]
running attemp: 0
2024:   7728 rows  tableRowSizes=[114, 114, 114, 114, 114, 74]
running attemp: 0
2025:   8784 rows  tableRowSizes=[130, 130, 130, 130, 130, 82]
running attemp: 0
2026:   3912 rows  tableRowSizes=[116, 116, 116, 116, 116, 72]
linhas long: 36468
HTS Number              str
Country                 str
District                str
Year                    str
Quantity Description    str
Measure                 str
Unit                    str
Month                   str
Value                   str
<StringArray>
[         'Landed Duty-Paid Value',                   'Customs Value',
 'Charges, Insurance, and Freight',               'Calculated Duties',
          'First Unit of Quantity',         'Second Unit of Quantity']
Length: 6, dtype: str


In [58]:
PIVOT_KEY = ["HTS Number", "Country", "District", "Year", "Month"]
MONEY_COLUMNS = {"Landed Duty-Paid Value": "landed_duty_paid",
                 "Customs Value": "customs_value",
                 "Charges, Insurance, and Freight": "charges_ins_freight",
                 "Calculated Duties": "calculated_duties"}
QUANTITY_COLUMNS = {"First Unit of Quantity": "qty_first",
                    "Second Unit of Quantity": "qty_second"}
MONTH_NUMBER = {m: i for i, m in enumerate(
    ["January", "February", "March", "April", "May", "June", "July",
     "August", "September", "October", "November", "December"], 1)}


In [59]:
def widen(old_df):
    df = old_df.copy()
    df["V"] = pd.to_numeric(df["Value"].str.replace(",", ""), errors="coerce").fillna(0)

    money = (df[df["Measure"].isin(MONEY_COLUMNS)]
             .groupby(PIVOT_KEY + ["Measure"])["V"].sum()
             .unstack("Measure").rename(columns=MONEY_COLUMNS)) # O measure deveria virar uma coluna, porque ele pe o que difere que valor é registrado no "Value"

    qty = df[df["Measure"].isin(QUANTITY_COLUMNS)].copy()
    qty["col"] = qty["Measure"].map(QUANTITY_COLUMNS) + "_" + qty["Quantity Description"]
    qty = qty.groupby(PIVOT_KEY + ["col"])["V"].sum().unstack("col")

    wide = money.join(qty, how="outer").fillna(0).reset_index()
    print(wide.columns.to_list())

    wide["kg_total"] = wide["qty_first_kilograms"] + wide["qty_second_kilograms"]

    before = len(wide)
    wide = wide[(wide["kg_total"] > 0) & (wide["landed_duty_paid"] > 0)].copy()
    print(f"linhas removidas (sem peso declarado ou sem valor): {before - len(wide)} de {before}")

    wide["date"] = pd.to_datetime(dict(year=wide["Year"].astype(int),
                                       month=wide["Month"].map(MONTH_NUMBER), day=1))
    return wide.sort_values(["HTS Number", "Country", "date", "District"])


df_wide = widen(df_dataweb)
df_wide.to_csv(os.path.join(os.getcwd(), "dataweb_wide.csv"), index=False)

identity = (df_wide["landed_duty_paid"]
            - (df_wide["customs_value"] + df_wide["charges_ins_freight"] + df_wide["calculated_duties"])).abs()
orphan = df_wide[(df_wide["landed_duty_paid"] > 0) & (df_wide["kg_total"] == 0)]

print(f"linhas: {len(df_wide)}  colunas: {list(df_wide.columns)}")
print(f"identidade landed == customs + frete + imposto, max diff: {identity.max()}")
print(f"linhas com valor e kg=0: {len(orphan)}  "
      f"(US$ {orphan['landed_duty_paid'].sum():,.0f} de {df_wide['landed_duty_paid'].sum():,.0f} = "
      f"{orphan['landed_duty_paid'].sum() / df_wide['landed_duty_paid'].sum() * 100:.4f}%)")
print(df_wide.head(3).to_string(index=False))


['HTS Number', 'Country', 'District', 'Year', 'Month', 'calculated_duties', 'charges_ins_freight', 'customs_value', 'landed_duty_paid', 'qty_first_kilograms', 'qty_first_number', 'qty_second_kilograms']
linhas removidas (sem peso declarado ou sem valor): 3192 de 6492
linhas: 3300  colunas: ['HTS Number', 'Country', 'District', 'Year', 'Month', 'calculated_duties', 'charges_ins_freight', 'customs_value', 'landed_duty_paid', 'qty_first_kilograms', 'qty_first_number', 'qty_second_kilograms', 'kg_total', 'date']
identidade landed == customs + frete + imposto, max diff: 0
linhas com valor e kg=0: 0  (US$ 0 de 3,710,289,058 = 0.0000%)
HTS Number Country        District Year   Month  calculated_duties  charges_ins_freight  customs_value  landed_duty_paid  qty_first_kilograms  qty_first_number  qty_second_kilograms  kg_total       date
    730630  Brazil     Chicago, IL 2022 January                  0                38088         211561            249649              81042.0               0.0 

In [60]:
ATRIBUICAO_MIN = 0.80      # o embarque precisa cobrir quase todo o peso da celula
ATRIBUICAO_MAX = 1.20      # acima disso o embarque excede a celula: distrito errado

COLS = ["District", "mes", "hs6", "Shipper", "kg_volza", "kg_total",
        "cobertura", "usd_por_kg", "atribuivel", "motivo"]


In [61]:
def atribuir_embarque(solo, wide, mini=ATRIBUICAO_MIN, maxi=ATRIBUICAO_MAX):
    #Se um embarque unico do Volza responde por quase todo o peso de uma celula
    #(distrito, mes, HTS, pais) do DataWeb, entao o US$/kg daquela celula pertence
    #ao shipper nomeado — e um preco com nome, que nenhuma das fontes da sozinha.
    #Hoje nenhuma linha qualifica; a logica fica pronta para quando qualificar.
    w = wide.copy()
    w["HTS Number"] = w["HTS Number"].astype(str)
    m = solo.rename(columns={"district_provavel": "District", "kg": "kg_volza"}).merge(
        w, left_on=["District", "mes", "hs6", "Country of Origin"],
        right_on=["District", "date", "HTS Number", "Country"], how="left")

    m["cobertura"] = m["kg_volza"] / m["kg_total"]
    m["usd_por_kg"] = m["landed_duty_paid"] / m["kg_total"]
    m["atribuivel"] = m["cobertura"].between(mini, maxi)
    m["motivo"] = np.where(m["kg_total"].isna(), "sem celula no DataWeb",
                  np.where(m["cobertura"] > maxi, "embarque excede a celula (distrito errado)",
                  np.where(m["cobertura"] < mini, "celula tem outra carga junto", "atribuivel")))
    return m.sort_values("cobertura", ascending=False)


atr = atribuir_embarque(df_solo, df_wide)
print(atr[COLS].round({"kg_volza": 0, "kg_total": 0, "cobertura": 3, "usd_por_kg": 2}).to_string(index=False))

print(f"\natribuiveis ({ATRIBUICAO_MIN:.0%}-{ATRIBUICAO_MAX:.0%} da celula): "
      f"{int(atr['atribuivel'].sum())} de {len(atr)}")
print(atr["motivo"].value_counts().to_string())

# Prova de que a logica dispara: o candidato mais proximo de cobertura 1 (em log,
# para nao privilegiar o lado de cima), com a faixa afrouxada so o bastante para
# alcanca-lo. Nao e resultado, e demonstracao de que o pipeline emite um preco.
c = atr[atr["cobertura"].notna()].copy()
melhor = c.loc[np.log(c["cobertura"]).abs().idxmin()]
folga = float(melhor["cobertura"])
demo = atribuir_embarque(df_solo, df_wide, mini=min(folga, 1 / folga) * 0.99,
                         maxi=max(folga, 1 / folga) * 1.01)
ok = demo[demo["atribuivel"]]
print(f"\n--- demonstracao: faixa afrouxada para {min(folga, 1/folga):.0%}-{max(folga, 1/folga):.0%} ---")
print(ok[["Shipper", "District", "mes", "kg_volza", "kg_total", "cobertura", "usd_por_kg"]]
      .round({"cobertura": 3, "usd_por_kg": 2}).to_string(index=False))
print(f"\nCom cobertura dentro da faixa, a saida seria: {melhor['Shipper']} a "
      f"US$ {melhor['usd_por_kg']:.2f}/kg em {melhor['District']}, {melhor['mes']:%Y-%m}.\n"
      f"Hoje esse numero descreve a celula inteira ({melhor['cobertura']:.0%} de cobertura), "
      f"nao o embarque.")

             District        mes    hs6                           Shipper  kg_volza  kg_total  cobertura  usd_por_kg  atribuivel                                     motivo
      Los Angeles, CA 2025-11-01 848210   CTL Logistics (India) Pvt. Ltd.     19500     282.0     69.149       33.93       False embarque excede a celula (distrito errado)
      Los Angeles, CA 2026-01-01 848210                 Z To Order and NA      3949     569.0      6.940       93.24       False embarque excede a celula (distrito errado)
Houston-Galveston, TX 2025-12-01 730630                 Z To Order and NA    184253   37557.0      4.906        0.97       False embarque excede a celula (distrito errado)
          Seattle, WA 2025-11-01 730630       Harbour Faith Logistics Inc     15060   40640.0      0.371        1.51       False               celula tem outra carga junto
          Seattle, WA 2026-01-01 730630    Tube Investments of India Ltd.     24353  108313.0      0.225        2.19       False            

## 7. Preco por quilo


In [62]:
SERIES_KEY = ["HTS Number", "Country", "date"]
PANEL_KEY = ["HTS Number", "Country", "District"]
SUM_COLUMNS = ["landed_duty_paid", "customs_value", "charges_ins_freight",
               "calculated_duties", "kg_total", "qty_first_number",
               "qty_second_kilograms"]
KG_PER_PIECE_TOLERANCE = 3.0
PRICE_TOLERANCE = 3.0
ROLLING_WINDOW = 3
MIN_KG_FOR_PRICE = 10_000
HTS_LABEL = {"730630": "7306.30 - tubos soldados, secao circular, aco nao ligado",
             "848210": "8482.10 - rolamentos de esferas"}


In [63]:
def aggregate_country(df):
    """Soma todos os distritos do pais. As razoes so sao calculadas depois da
    soma: media ponderada e soma(numerador)/soma(denominador), nunca a media
    das razoes por distrito."""
    totals = df.groupby(SERIES_KEY, as_index=False)[SUM_COLUMNS].sum()
    totals = totals[totals["kg_total"] > 0].copy()

    totals["usd_per_kg_landed"] = totals["landed_duty_paid"] / totals["kg_total"]
    totals["usd_per_kg_customs"] = totals["customs_value"] / totals["kg_total"]
    totals["freight_share"] = totals["charges_ins_freight"] / totals["landed_duty_paid"]
    totals["duty_share"] = totals["calculated_duties"] / totals["landed_duty_paid"]
    totals["effective_duty_rate"] = totals["calculated_duties"] / totals["customs_value"]
    totals["commodity"] = totals["HTS Number"].astype(str).map(HTS_LABEL)
    totals["kg_per_piece"] = (totals["qty_second_kilograms"]
                              / totals["qty_first_number"].replace(0, pd.NA))
    return totals.sort_values(SERIES_KEY)


def flag_district_outliers(df, reference):
    """Referencia = mediana movel de ROLLING_WINDOW registros da serie nacional
    (nao e janela de calendario: distritos e meses tem buracos). Cada distrito-mes
    e julgado contra ela sem ser agregado."""
    ref = reference.sort_values(SERIES_KEY).copy()
    ref["baseline"] = (ref.groupby(["HTS Number", "Country"])["usd_per_kg_landed"]
                       .transform(lambda s: s.rolling(ROLLING_WINDOW, min_periods=1,
                                                      center=True).median()))

    panel = df.copy()
    panel["usd_per_kg_landed"] = panel["landed_duty_paid"] / panel["kg_total"]
    panel = panel.merge(ref[SERIES_KEY + ["baseline"]], on=SERIES_KEY, how="left")
    panel["price_ratio"] = panel["usd_per_kg_landed"] / panel["baseline"]
    panel["price_outlier"] = ((panel["price_ratio"] > PRICE_TOLERANCE)
                              | (panel["price_ratio"] < 1 / PRICE_TOLERANCE))
    return panel.sort_values(PANEL_KEY + ["date"])


def price_series(df):
    reference = aggregate_country(df)
    panel = flag_district_outliers(df, reference)
    totals = aggregate_country(panel[~panel["price_outlier"]])

    baseline = totals.groupby(["HTS Number", "Country"])["kg_per_piece"].transform("median")
    totals["implausible_unit"] = totals["kg_per_piece"] > KG_PER_PIECE_TOLERANCE * baseline
    totals["low_volume"] = totals["kg_total"] < MIN_KG_FOR_PRICE
    totals["reliable"] = ~(totals["implausible_unit"].fillna(False) | totals["low_volume"])
    return totals.sort_values(SERIES_KEY), panel


df_price, df_panel = price_series(df_wide)
df_price.to_csv(os.path.join(os.getcwd(), "dataweb_price_series.csv"), index=False)
df_panel.to_csv(os.path.join(os.getcwd(), "dataweb_price_districts.csv"), index=False)

outliers = df_panel[df_panel["price_outlier"]]
print(f"painel distrital: {len(df_panel)} linhas  "
      f"({df_panel['District'].nunique()} distritos)")
print(f"outliers de preco (fora de {1 / PRICE_TOLERANCE:.2f}x a {PRICE_TOLERANCE:.2f}x "
      f"a mediana movel de {ROLLING_WINDOW} registros da serie nacional): {len(outliers)}  "
      f"(US$ {outliers['landed_duty_paid'].sum():,.0f} de "
      f"{df_panel['landed_duty_paid'].sum():,.0f} = "
      f"{outliers['landed_duty_paid'].sum() / df_panel['landed_duty_paid'].sum() * 100:.4f}%)")
print(f"series: {len(df_price)} linhas  ({df_price['HTS Number'].nunique()} HTS x "
      f"{df_price['Country'].nunique()} paises x {df_price['date'].nunique()} meses)")
print(f"periodo: {df_price['date'].min():%Y-%m} a {df_price['date'].max():%Y-%m}")

print(f"\n=== outliers distritais removidos (20 maiores de {len(outliers)}) ===")
print(outliers.nlargest(20, "price_ratio")[
    ["HTS Number", "Country", "District", "date", "kg_total",
     "usd_per_kg_landed", "baseline", "price_ratio"]]
    .round({"kg_total": 1, "usd_per_kg_landed": 2, "baseline": 2, "price_ratio": 1})
    .to_string(index=False))

print("\n=== US$/kg landed, mediana por HTS x pais ===")
print(df_price.pivot_table(index="Country", columns="HTS Number",
                           values="usd_per_kg_landed", aggfunc="median").round(2).to_string())

print("\n=== composicao do custo (media ponderada, so celulas confiaveis) ===")
comp = df_price[df_price["reliable"]].groupby(["HTS Number", "Country"]).apply(
    lambda g: pd.Series({
        "usd_kg_landed": g["landed_duty_paid"].sum() / g["kg_total"].sum(),
        "usd_kg_mercadoria": g["customs_value"].sum() / g["kg_total"].sum(),
        "usd_kg_frete": g["charges_ins_freight"].sum() / g["kg_total"].sum(),
        "usd_kg_imposto": g["calculated_duties"].sum() / g["kg_total"].sum(),
        "aliquota_efetiva_%": g["calculated_duties"].sum() / g["customs_value"].sum() * 100}),
    include_groups=False).round(2)
print(comp.to_string())

print("\n=== sanidade ===")
print(f"celulas com peso implausivel (>{KG_PER_PIECE_TOLERANCE}x a mediana da serie): "
      f"{int(df_price['implausible_unit'].fillna(False).sum())}")
print(f"celulas de volume baixo (<{MIN_KG_FOR_PRICE:,} kg): {int(df_price['low_volume'].sum())}")
print(f"celulas confiaveis: {int(df_price['reliable'].sum())} de {len(df_price)}")
print(df_price[~df_price["reliable"]][["HTS Number", "Country", "date", "kg_total",
                                       "kg_per_piece", "usd_per_kg_landed"]].to_string(index=False))

painel distrital: 3300 linhas  (39 distritos)
outliers de preco (fora de 0.33x a 3.00x a mediana movel de 3 registros da serie nacional): 929  (US$ 217,569,342 de 3,710,289,058 = 5.8639%)
series: 390 linhas  (2 HTS x 4 paises x 54 meses)
periodo: 2022-01 a 2026-06

=== outliers distritais removidos (20 maiores de 929) ===
HTS Number Country              District       date  kg_total  usd_per_kg_landed  baseline  price_ratio
    848210  Mexico           El Paso, TX 2024-01-01       2.0           52800.00     15.23       3465.9
    848210  Brazil        Wilmington, NC 2022-05-01       2.0           46233.00     18.68       2475.2
    848210  Canada          New York, NY 2024-01-01       1.0           56463.00     26.91       2098.0
    848210  Canada Dallas-Fort Worth, TX 2022-11-01       3.0           61604.00     30.27       2035.2
    848210   India           Buffalo, NY 2024-09-01       1.0           15459.00     19.58        789.7
    848210  Mexico         Cleveland, OH 2025-12-01 

## 8. Scorecard por pais

Nivel de preco na janela pos-tarifa, risco por horizonte e cobertura do dado. Series com furo nao sao removidas: a imprecisao delas e reportada.


In [66]:
POST_TARIFF = pd.Timestamp("2025-04-01")   # mes do Liberation Day, marco 1 do grafico
HORIZONS = [1, 3, 6, 12]
MIN_PAIRS = 8   # o erro relativo de um desvio-padrao estimado e 1/sqrt(2(n-1)):
                # com 8 pares ele ja e 19%, com 3 pares e 50%. Abaixo de 8 o sigma
                # nao mede risco, mede o tamanho da amostra. Ver err_sigma_%.


In [67]:
def longest_gap(mask):
    off = (~mask).astype(int)
    return int(off.groupby((off != off.shift()).cumsum()).sum().max())


def scorecard(price):
    months = pd.date_range(price["date"].min(), price["date"].max(), freq="MS")
    rows = []
    for (hts, country), d in price[price["reliable"]].groupby(["HTS Number", "Country"]):
        logp = np.log(d.set_index("date")["usd_per_kg_landed"].reindex(months))
        post = d[d["date"] >= POST_TARIFF]
        n = len(post)
        # nivel: identidade contabil US$/kg, nunca media de razoes
        nivel = post["landed_duty_paid"].sum() / post["kg_total"].sum() if n else np.nan
        sd = post["usd_per_kg_landed"].std(ddof=1) if n > 1 else np.nan
        pares = int(logp.diff(1).notna().sum())        # so meses consecutivos
        row = {"HTS Number": hts, "Country": country, "usd_kg": nivel, "n_meses_pos": n,
               # erro padrao da media; subestima onde a serie e passeio aleatorio
               "ep_nivel_%": sd / np.sqrt(n) / nivel * 100 if n > 1 else np.nan,
               "cobertura_%": logp.notna().mean() * 100,
               "maior_furo": longest_gap(logp.notna()),
               "n_pares": pares,
               # o quanto o sigma erra: 1/sqrt(2(n-1)), o criterio por tras de MIN_PAIRS
               "err_sigma_%": 100 / np.sqrt(2 * (pares - 1)) if pares > 1 else np.nan,
               "duty_share_%": (post["calculated_duties"].sum()
                                / post["landed_duty_paid"].sum() * 100) if n else np.nan}
        for h in HORIZONS: # furos viram NaN e caem no dropna
            dif = logp.diff(h).dropna()
            row[f"sigma_h{h}_%"] = dif.std(ddof=1) * 100 if len(dif) >= MIN_PAIRS else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index(["HTS Number", "Country"]).sort_index()


df_score = scorecard(df_price)
df_score.to_csv(os.path.join(os.getcwd(), "dataweb_scorecard.csv"))

nivel = ["usd_kg", "ep_nivel_%", "n_meses_pos", "duty_share_%"]
risco = [f"sigma_h{h}_%" for h in HORIZONS]
cober = ["cobertura_%", "maior_furo", "n_pares", "err_sigma_%"]

print(f"nivel de preco, janela pos-tarifa desde {POST_TARIFF:%Y-%m}")
print(df_score[nivel].round(2).to_string())
print("\nvariacao esperada em h meses, 1 sigma %")
print(df_score[risco].round(1).to_string())
print("\ncobertura e confianca da estimativa")
print(df_score[cober].round(1).to_string())
print(f"\nNaN em sigma_h = menos de {MIN_PAIRS} pares utilizaveis; a linha fica, "
      f"a estimativa nao. Ver cobertura_% e maior_furo.")

nivel de preco, janela pos-tarifa desde 2025-04
                    usd_kg  ep_nivel_%  n_meses_pos  duty_share_%
HTS Number Country                                               
730630     Brazil     1.86       10.84            3         28.86
           Canada     1.74        1.52           15         29.54
           India      1.96        1.64           15         28.56
           Mexico     2.33        1.97           15         28.43
848210     Brazil    21.79        7.10           12         26.29
           Canada    33.19        5.62           15         11.61
           India     20.00        4.97           15         25.89
           Mexico    14.12        3.17           14         13.51

variacao esperada em h meses, 1 sigma %
                    sigma_h1_%  sigma_h3_%  sigma_h6_%  sigma_h12_%
HTS Number Country                                                 
730630     Brazil          NaN         NaN         NaN          NaN
           Canada          6.6        12.3     

## 9. Volza x ImportYeti: concordancia

As duas fontes leem o mesmo manifesto maritimo da CBP, entao divergencia entre elas
e diferenca de amostra, nao de fato. Como o ranking de exportadores da secao 4c sai
so do Volza, o tamanho dessa amostra precisa estar medido ao lado dele.

O teste e contar os embarques de cada exportador nos dois lados, na mesma janela.
As barras de fornecedor do ImportYeti sao trimestrais (so meses 1, 4, 7 e 10), entao
a janela do Volza — 10/11/2025 a 04/02/2026 — so pode ser arredondada para fora, para
Q4/2025 e Q1/2026. Por isso a razao total sai abaixo de 1 por construcao: o que
interessa e quem aparece nos dois lados e se a ordem bate.

In [71]:
VALID_INICIO = pd.Timestamp("2025-10-01")
VALID_FIM = pd.Timestamp("2026-03-31")

# A marca e a chave de juncao: "Schaeffler Brasil Ltda" (Volza) e "Schaeffler
# Brasil" (ImportYeti) viram o mesmo frozenset depois que marca() tira forma
# juridica e pais. Comparar a string crua nao casaria nenhum dos dois.
v = pd.read_csv(VOLZA_CSV)
v["hs6"] = v["HS Code"].astype(str).str[:6]
v["chave"] = [frozenset(marca(s)) for s in v["Shipper"]]
volza = (v[pd.to_datetime(v["Date"]).between(VALID_INICIO, VALID_FIM)]
         .groupby(["hs6", "chave"]).size())

y = pd.concat([pd.read_csv(os.path.join(os.getcwd(), f"importyeti_supplier_shipments_{h}.csv")).assign(hs6=h)
               for h in ["730630", "848210"]])
y["chave"] = [frozenset(marca(n)) for n in y["name"]]
yeti = (y[pd.to_datetime(y["date"], format="%b %Y").between(VALID_INICIO, VALID_FIM)]
        .groupby(["hs6", "chave"])["shipments"].sum())

conc = pd.concat([yeti.rename("yeti"), volza.rename("volza")], axis=1).fillna(0).astype(int)
conc["exportador"] = [", ".join(sorted(k)) or "(sem marca)" for _, k in conc.index]
print(conc.sort_values("yeti", ascending=False).to_string())

ambos = conc[(conc["yeti"] > 0) & (conc["volza"] > 0)]
print(f"\ntotais: yeti {conc['yeti'].sum()}  volza {conc['volza'].sum()}  "
      f"razao {conc['volza'].sum() / conc['yeti'].sum():.2f}")
print(f"nos dois: {len(ambos)}   so ImportYeti: {int((conc['volza'] == 0).sum())}   "
      f"so Volza: {int((conc['yeti'] == 0).sum())}")
print(f"spearman entre as contagens dos que aparecem nos dois: "
      f"{ambos[['yeti', 'volza']].corr(method='spearman').iloc[0, 1]:.2f}")

                                                 yeti  volza                   exportador
hs6    chave                                                                             
730630 frozenset({goodluck})                      298      0                     goodluck
848210 frozenset({schaeffler})                    201    109                   schaeffler
730630 frozenset({investments})                   149      1                  investments
       frozenset({dahnay})                         48      0                       dahnay
       frozenset({metamorphosis, engitech})        45      3      engitech, metamorphosis
848210 frozenset({priv, agrasen})                  38      0                agrasen, priv
       frozenset({nrb})                            24      0                          nrb
730630 frozenset({surya, roshni})                  24      0                roshni, surya
       frozenset({smith, privat})                  23      0                privat, smith
       fro

## 10. O que o dado nao cruza

As duas fontes deste caderno nascem de registros diferentes e nao se encontram no
meio.

**DataWeb (USITC)** — estatistica oficial de comercio exterior dos EUA, montada a
partir das declaracoes de entrada apresentadas a CBP e consolidada pelo Census
Bureau. Vem agregada: HTS x pais x distrito aduaneiro x mes. Tem valor aduaneiro,
frete, seguro e imposto calculado. **Nao tem nome de empresa** — estatistica
agregada nao publica importador nem exportador.

**Volza e ImportYeti** — os dois leem o mesmo material: o manifesto maritimo de
importacao (bill of lading) que a CBP divulga. Vem por embarque: shipper,
consignee, porto, peso, descricao da mercadoria. **Nao tem valor** — o manifesto
declara carga, nao preco.

Um lado tem preco sem nome, o outro tem nome sem preco. Quatro coisas impedem a
juncao:

1. **Unidade de observacao.** A celula do DataWeb agrega todos os embarques de um
   distrito num mes; a linha do Volza e um conhecimento de embarque. Atribuir o
   preco da celula a um shipper so e legitimo se aquele shipper for praticamente a
   celula inteira — e a regra de 80% a 120% do capitulo 6. Nenhuma linha qualifica.
2. **Modal.** O manifesto publicado e maritimo. Mexico e Canada entram nos EUA por
   caminhao e trem, que o DataWeb registra e o manifesto nao. Os dois paises de
   nearshoring sao justamente os mais subrepresentados aqui: 1 e 11 embarques no
   Volza, contra 88 da India.
3. **Janela.** O Volza acessivel cobre 2,8 meses (11/2025 a 02/2026, 151
   embarques); o DataWeb cobre 2022 a 2026 em base mensal. Nao ha historico comum
   onde casar as duas.
4. **Chave geografica.** O manifesto traz porto de descarga, o DataWeb traz
   distrito aduaneiro — divisoes diferentes. O `PORTO_DISTRITO` do capitulo 4b e um
   mapa manual e aproximado.

Por isso as duas fontes aparecem lado a lado no capitulo 4c, e nao numa tabela so.
O cruzamento esta implementado e testado; ele simplesmente nao dispara com o dado
disponivel hoje.